# DiCo-NLI multilingual pilot (Tracks 2–4) — Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Pilot runs with **mDeBERTa-v3-base**, one seed (42), same recipe as the A2 runs (fp32, best epoch picked on the validation split, which holds out the same source pairs as A2).

| Run | Train on | Evaluate on | Question it answers |
|---|---|---|---|
| X0 | EN | EN, ES, EU, MIX | Cost of a multilingual encoder on English (vs RoBERTa 0.765) + zero-shot transfer |
| X1 | ES | ES | In-language Spanish |
| X2 | EU | EU | In-language Basque (low-resource) |
| X3 | EN+ES+EU | EN, ES, EU, MIX | Does pooling languages help? Zero-shot on mixed pairs |
| X4 *(optional, ~30 min)* | MIX | MIX | Proper Track 4 system |

Runs that already finished are skipped, so after a disconnect just run the cells again.

In [ ]:
# 1. Setup: clone the team repo (code, Track 1 data, official scorer)
!git clone -q https://github.com/MuneebBaig71/DiCo-NLI_SemEval.git
%cd DiCo-NLI_SemEval
!pip -q install sentencepiece protobuf
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Data for Tracks 2-4. Your repo only has Track 1, so copy the organizers'
#    final_data/ into data/ (cp -n never overwrites files that already exist).
#    If you have committed the Track 2-4 files to the repo, this does nothing.
import os
if not os.path.exists("data/train/dico_nli_train_track2_reference.csv"):
    !git clone -q --depth 1 https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git /content/dico_official
    !cp -n /content/dico_official/final_data/train/*.csv data/train/
    !cp -n /content/dico_official/final_data/dev/*.csv data/dev/

from multilingual.ml_data import TRACKS, load, check_data
check_data()
for t in TRACKS:
    tr, dv = load(t, "train"), load(t, "dev")
    print(f"{t:4s} train {len(tr):6d}  dev {len(dv):5d}  dev labels {dv.label.value_counts().to_dict()}")

In [ ]:
# 3. (Recommended) save results to Google Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/DiCo_multilingual_results"

In [ ]:
# 4. Smoke test (~2-3 min): 1 epoch of mDeBERTa on 300 Spanish rows, scored on ES and MIX.
#    Checks: model downloads, no NaN, files written, scorer runs on Tracks 2 and 4. Not a real result.
import importlib, multilingual.ml_data, multilingual.ml_train, multilingual.ml_eval
for m in (multilingual.ml_data, multilingual.ml_train, multilingual.ml_eval): importlib.reload(m)
from multilingual.ml_train import run
from multilingual.ml_eval import score_all
run(train_tracks=["es"], eval_tracks=["es", "mix"], epochs=1, max_train_rows=300,
    tag="smoke", out_dir="results_ml_smoke")
res, _, _ = score_all("results_ml_smoke")
res[["eval_on", "decoding", "wf1", "soft_cons", "hard_cons"]].round(3)

In [ ]:
# 5. Main pilot runs, seed 42 (about 10 min each on a T4; X3 about 30 min)
import shutil
from pathlib import Path
from multilingual.ml_train import run

MODEL = "microsoft/mdeberta-v3-base"
SEED = 42
ALL = ["en", "es", "eu", "mix"]
EXPS = [
    dict(tag="X0_en",     train_tracks=["en"],             eval_tracks=ALL),
    dict(tag="X1_es",     train_tracks=["es"],             eval_tracks=["es"]),
    dict(tag="X2_eu",     train_tracks=["eu"],             eval_tracks=["eu"]),
    dict(tag="X3_joint",  train_tracks=["en", "es", "eu"], eval_tracks=ALL),
]

def run_all(exps):
    for e in exps:
        tag = f"{e['tag']}_seed{SEED}"
        if Path(f"results_multilingual/runs/{tag}.json").exists():
            print("skip (done):", tag); continue
        run(**{**e, "tag": tag, "model": MODEL, "seed": SEED})
        try:   # copy after every run, so a disconnect loses at most one run
            shutil.copytree("results_multilingual", DRIVE_OUT, dirs_exist_ok=True)
        except NameError:
            pass

run_all(EXPS)

In [ ]:
# 6. (Optional) Track 4 trained on mixed pairs. 18k rows, so 3 epochs (~30 min).
run_all([dict(tag="X4_mix", train_tracks=["mix"], eval_tracks=["mix"], epochs=3)])

In [ ]:
# 7. Score everything on dev with the official scorer (argmax, symmetric, struct_pair)
!python -m multilingual.ml_eval

In [ ]:
# 8. Main table for the report (argmax = the model itself, struct_pair = A2 decoding)
import pandas as pd
res = pd.read_csv("results_multilingual/ml_results.csv")
cols = ["run", "train_on", "eval_on", "setting", "decoding", "wf1", "soft_cons", "hard_cons", "macro_f1"]
res[res.decoding.isin(["argmax", "struct_pair"])][cols].round(3)

In [ ]:
# 9. Training summary (best epoch chosen on the validation split, not dev)
import json
rows = []
for f in sorted(Path("results_multilingual/runs").glob("*.json")):
    m = json.loads(f.read_text())
    rows.append(dict(tag=m["tag"], train=" + ".join(m["cfg"]["train_tracks"]), best_epoch=m["best_epoch"],
                     best_val_wf1=round(m["best_val_wf1"], 4), minutes=round(m["seconds"] / 60, 1)))
pd.DataFrame(rows)

In [ ]:
# 10. Download everything as a zip (unzip into your repo folder on the Mac)
from google.colab import files
try:
    shutil.copytree("results_multilingual", DRIVE_OUT, dirs_exist_ok=True)
except NameError:
    pass
!zip -qr results_multilingual.zip results_multilingual
files.download("results_multilingual.zip")